# Phase 01: Dataset Exploration - Code Mentor AI

In this notebook, we explore the **`ise-uiuc/Magicoder-Evol-Instruct-110K`** dataset. Our goal is to examine its structure, check data quality, and verify its suitability for fine-tuning our **Code Mentor AI** assistant (focusing on code generation, explanation, and debugging).

### Notebook Outline:
1. Environment Setup & Dataset Load
2. Dataset Structure & Columns Inspection
3. Manual Sample Examination
4. Missing Values, Duplicates & Data Quality Check
5. Task Coverage & Response Length Analysis
6. Final Suitability Report

In [1]:
#import libraries 
import pandas as pd 
import random
from datasets import load_dataset

In [2]:
#load the dataset using hugging face
ds = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K")

README.md:   0%|          | 0.00/394 [00:00<?, ?B/s]

data-evol_instruct-decontaminated.jsonl: reconstructing file:   0%|          |  0.00B /  255MB            

data-evol_instruct-decontaminated.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/111183 [00:00<?, ? examples/s]

In [3]:
dataset = ds['train']

print(f"Dataset successfully loaded!")
print(ds)
print(f"Total rows in train split: {len(dataset):,}")


Dataset successfully loaded!
DatasetDict({
    train: Dataset({
        features: ['instruction', 'response'],
        num_rows: 111183
    })
})
Total rows in train split: 111,183


In [4]:
dataset

Dataset({
    features: ['instruction', 'response'],
    num_rows: 111183
})

### Step 2: Dataset Structure & Column Inspection
Checking the features and schema dynamically to verify column names.

In [5]:
#step 2: Dataset Structure & Dynamic column detection
all_columns = dataset.column_names
print(f"Available columns in dataset: {all_columns}")
print("\nDataset Feature/Schema: ")
print(dataset.features)

#Dynamic mapping to ensure safety 
prompt_col = 'instruction' if 'instruction' in all_columns else all_columns[0]
response_col = 'response' if 'response' in all_columns else all_columns[1]

print(f"\n[+] Active Instruction Column: '{prompt_col}'")
print(f"[+] Active Response Column: '{response_col}'")

Available columns in dataset: ['instruction', 'response']

Dataset Feature/Schema: 
{'instruction': Value('string'), 'response': Value('string')}

[+] Active Instruction Column: 'instruction'
[+] Active Response Column: 'response'


### Step 3: Manual Sample Examination
Inspecting random samples to evaluate the quality of instruction phrasing, explanations, and code formatting.

In [6]:
#step 3: Manual Sample Examination
sample_indices = random.sample(range(len(dataset)),3)
for i, idx in enumerate(sample_indices, 1):
    sample = dataset[idx]
    print(f"\n{'='*20} Sample {i} (Index: {idx}) {'='*20}")
    print(f"[{prompt_col.upper()}] (Instruction):\n{str(sample.get(prompt_col, ''))[:600]}...\n")
    print(f"[{response_col.upper()}] (Code/Explanation):\n{str(sample.get(response_col, ''))[:600]}...\n")


==================== Sample 1 (Index: 53412) ====================
[INSTRUCTION] (Instruction):
Detail four distinct, fully fleshed-out class structures ideal for implementing a technologically advanced application specifically tailored for electronic banking operations....

[RESPONSE] (Code/Explanation):
Below are four basic class structures i.e. `Customer`, `Account`, `Transaction` and `Bank` that can be suitable for implementing a technologically advanced electronic banking operations.

1. Customer Class:
This class would represent a customer at the bank. Customers of the bank could be both individuals or businesses. 

```python
class Customer:
    def __init__(self, name, address, email, phone):
        self.name = name
        self.address = address
        self.email = email
        self.phone = phone
```

2. Account Class:
This holds details about all kinds of accounts including checking...


==================== Sample 2 (Index: 59329) ====================
[INSTRUCTION] (Instru

### Step 4: Missing Values, Duplicates & Data Quality Check
Converting to a pandas DataFrame to run structural checks for missing, empty, or duplicate entries.

In [7]:
#step 4: Missing values, duplicates & data quality check
df = dataset.to_pandas()
missing_prompt = df[prompt_col].isnull().sum()
missing_response = df[response_col].isnull().sum()
empty_prompt = (df[prompt_col].astype(str).str.strip() == "").sum()
empty_response = (df[response_col].astype(str).str.strip() == "").sum()
duplicates = df.duplicated(subset=[prompt_col]).sum()

print(f"Missing values in '{prompt_col}': {missing_prompt}")
print(f"Missing values in '{response_col}': {missing_response}")
print(f"Empty strings in '{prompt_col}': {empty_prompt}")
print(f"Empty strings in '{response_col}': {empty_response}")
print(f"Duplicate instruction entries: {duplicates}")

Missing values in 'instruction': 0
Missing values in 'response': 0
Empty strings in 'instruction': 0
Empty strings in 'response': 0
Duplicate instruction entries: 0


### Step 5: Task Coverage & Response Length Analysis
Analyzing text lengths to help decide the optimal `max_seq_length` token limit for fine-tuning.

In [8]:
# Step 5: Task Coverage & Response Length Analysis
df['prompt_word_len'] = df[prompt_col].astype(str).apply(lambda x: len(x.split()))
df['response_word_len'] = df[response_col].astype(str).apply(lambda x: len(x.split()))

print("--- Instruction / Prompt Word Count Distribution ---")
print(df['prompt_word_len'].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]))

print("\n--- Response / Output Word Count Distribution ---")
print(df['response_word_len'].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]))

--- Instruction / Prompt Word Count Distribution ---
count    111183.000000
mean         93.923621
std         143.160246
min           1.000000
50%          62.000000
75%         112.000000
90%         178.000000
95%         243.000000
99%         500.000000
max        2862.000000
Name: prompt_word_len, dtype: float64

--- Response / Output Word Count Distribution ---
count    111183.000000
mean        224.925375
std         102.639129
min           1.000000
50%         216.000000
75%         287.000000
90%         363.000000
95%         413.000000
99%         498.000000
max         946.000000
Name: response_word_len, dtype: float64


### Step 6: Final Suitability Report
Summarizing our data metrics to confirm readiness for the preprocessing phase.

In [9]:
# Step 6: Final Suitability Report
total_samples = len(df)
valid_samples = total_samples - (missing_prompt + missing_response + empty_prompt + empty_response)

report = {
    "Dataset Source": "ise-uiuc/Magicoder-Evol-Instruct-110K",
    "Total Rows": total_samples,
    "Valid Clean Rows": valid_samples,
    "Data Quality (%)": f"{(valid_samples / total_samples) * 100:.2f}%",
    "Duplicate Entries": duplicates,
    "Avg Prompt Length (Words)": int(df['prompt_word_len'].mean()),
    "Avg Response Length (Words)": int(df['response_word_len'].mean()),
    "95th Percentile Response Length (Words)": int(df['response_word_len'].quantile(0.95)),
    "Suitability Status": "HIGHLY SUITABLE for Code Generation, Explanation & Debugging Assistant"
}

print("=" * 60)
print("FINAL DATASET SUITABILITY REPORT")
print("=" * 60)
for key, val in report.items():
    print(f" • {key}: {val}")
print("=" * 60)

FINAL DATASET SUITABILITY REPORT
 • Dataset Source: ise-uiuc/Magicoder-Evol-Instruct-110K
 • Total Rows: 111183
 • Valid Clean Rows: 111183
 • Data Quality (%): 100.00%
 • Duplicate Entries: 0
 • Avg Prompt Length (Words): 93
 • Avg Response Length (Words): 224
 • 95th Percentile Response Length (Words): 413
 • Suitability Status: HIGHLY SUITABLE for Code Generation, Explanation & Debugging Assistant


# Conclusion: Phase 01 Dataset Exploration
Your first notebook successfully establishes a rigorous foundation for Code Mentor AI. By dynamically inspecting and validating the `ise-uiuc/Magicoder-Evol-Instruct-110K dataset`, you have ensured that your training pipeline starts with clean, high-utility data.

## Summary of Key Findings
`Dataset Scale & Integrity:` Confirmed a robust volume of 111,183 rows with clean structures across the instruction and response schema, ensuring high data quality with minimal noise.

`Instructional Diversity:` Manual sample inspection validated that the prompts and responses contain clear, step-by-step coding logic, explanations, and well-formatted code blocks crucial for a mentor persona.

`Context Window Baseline:` The response length distribution and 95th percentile analysis give you the exact metric needed to configure your max_seq_length safely during the upcoming tokenization phase without prematurely cutting off important code blocks.
